# gVXR aluminium sphere: polychromatic Beer–Lambert validation

This notebook checks a gVXR projection against an analytical **polychromatic** transmission calculation. A weighted set of photon-energy bins passes through a centered aluminium sphere. At the center pixel, each energy is attenuated by its own coefficient, so the reference is the spectrum-weighted sum of transmissions, followed by `−ln(I/I₀)`. A single `μ × diameter` is not the right reference for a polychromatic beam.

The example spectrum is illustrative and supplied as relative bin weights; it is not a measured or calibrated tube spectrum. The notebook also plots beam hardening: effective attenuation changes with path length as lower-energy photons are preferentially removed.

## 1. Provide the simulation core
Upload the project’s current `ct_core.py` if it is not already present in the Colab working directory.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content')
if not (PROJECT_DIR / 'ct_core.py').exists():
    from google.colab import files
    uploaded = files.upload()  # upload ct_core.py
    if 'ct_core.py' not in uploaded:
        raise FileNotFoundError('Please upload ct_core.py')


## 2. Install dependencies
Run once in this Colab runtime. `ct_core.py` configures the best available EGL/OpenGL renderer before creating the gVXR context.

In [ ]:
import subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'build-essential', 'libxcb-res0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxcb-icccm4', 'libxcb-image0',
                'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-shape0', 'libxcb-randr0', 'libxkbcommon-x11-0',
                'libxrandr2', 'libxinerama1', 'libxcursor1', 'libxi6', 'libxxf86vm1',
                'libgl1', 'libegl1', 'libopengl0', 'libglu1-mesa', 'libgl1-mesa-dri', 'libegl-mesa0', 'libglx-mesa0',
                'libxcb-composite0', 'libxcb-cursor0', 'libxcb-dri2-0', 'libxcb-ewmh2',
                'libxcb-dri3-0', 'libxcb-glx0', 'libxcb-present0', 'libxcb-sync1', 'libxcb-xfixes0', 'libxcb-xinput0',
                'libxcb-util1', 'libxcb-xv0', 'libxcb-damage0', 'libxcb-dpms0', 'libxcb-record0', 'libxcb-screensaver0',
                'libxcb-shm0', 'libxcb-xtest0', 'libxcb-xf86dri0', 'libxcb-xvmc0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gvxr', 'numpy', 'trimesh', 'xraylib', 'matplotlib'], check=True)


## 3. Set up a centered aluminium sphere
The odd number of detector columns gives a pixel centered on the beam axis. The 0.1 mm detector pitch samples the sphere finely. The provided multi-bin spectrum is an example chosen to demonstrate the polychromatic calculation; edit its weights to study other spectra.

In [ ]:
import json, os, platform, sys, tempfile
import numpy as np
import trimesh
import xraylib
import matplotlib.pyplot as plt
sys.path.insert(0, str(PROJECT_DIR.resolve()))
import ct_core as core

# Material parameters
RADIUS_MM = 5.0
DIAMETER_MM = 2 * RADIUS_MM
DENSITY_G_CM3 = 2.70
# Illustrative relative photon counts per energy bin; not a measured tube spectrum.
SPECTRUM_WEIGHTS = {30: 0.20, 40: 0.45, 50: 0.70, 60: 0.90, 70: 1.00,
                    80: 1.00, 90: 0.90, 100: 0.75, 110: 0.55, 120: 0.35}

# Detector and cone-beam setup
NX, NY = 401, 3  # odd NX puts a pixel at the central ray
PIXEL_MM = 0.1
SOD_CM, ODD_CM = 40.0, 10.0
MAGNIFICATION = (SOD_CM + ODD_CM) / SOD_CM

mesh_path = Path(tempfile.gettempdir()) / 'gvxr_aluminium_polychromatic_sphere.stl'
trimesh.creation.icosphere(subdivisions=4, radius=RADIUS_MM).export(mesh_path)
obj = core.ObjectCfg(name='aluminium_sphere', stl_path=str(mesh_path),
                     kind='Element', material='Al', density=DENSITY_G_CM3)
cfg = core.ScanCfg(objects=[obj], stl_unit='mm', beam='Cone',
                   source_x_cm=SOD_CM, detector_x_cm=ODD_CM,
                   energy_kev=80.0, photons=10000,
                   nx=NX, ny=NY, pixel_mm=PIXEL_MM,
                   n_angles=1, total_deg=360.0, noise=False, spectrum=SPECTRUM_WEIGHTS)

flat = core.on_gl_thread(core.build_scene, cfg)
raw = core.on_gl_thread(lambda: np.asarray(core._gx().computeXRayImage(), dtype=np.float32))
print('Projection shape:', raw.shape)
print('OpenGL renderer:', core.renderer_info())
print('Renderer selection:', core.renderer_selection_info())
print('Relative spectrum weights:', SPECTRUM_WEIGHTS)


## 4. Compare measured attenuation with the spectrum-integrated reference
For energy bin `Eᵢ`, primary transmission through length `L` is `exp(−μᵢL)`. The project’s image is energy-weighted, so the expected intensity ratio uses each bin’s photon count times its energy: `I/I₀ = Σ(nᵢ Eᵢ exp(−μᵢL)) / Σ(nᵢ Eᵢ)`. Then measured/reference attenuation is `−ln(I/I₀)`. The line profile uses the sphere chord length at each object-space offset.

In [ ]:
# Match the energy-bin counts used by ct_core.
bin_counts = core._spectrum_counts(cfg)
energies_kev = np.asarray(list(bin_counts.keys()), dtype=float)
counts = np.asarray([bin_counts[e] for e in energies_kev], dtype=float)
mass_mu_cm2_g = np.asarray([xraylib.CS_Total(xraylib.SymbolToAtomicNumber('Al'), e)
                            for e in energies_kev], dtype=float)
mu_bins_cm_inv = mass_mu_cm2_g * DENSITY_G_CM3
energy_fluence_weights = counts * energies_kev  # photons × keV, as in core.flat_value()
diameter_cm = DIAMETER_MM / 10.0

def poly_attenuation(path_cm):
    path_cm = np.asarray(path_cm, dtype=float)
    transmitted = np.sum(energy_fluence_weights[:, None] *
                         np.exp(-mu_bins_cm_inv[:, None] * path_cm.reshape(1, -1)), axis=0)
    incident = np.sum(energy_fluence_weights)
    return -np.log(np.maximum(transmitted / incident, np.finfo(float).tiny))

expected_center = float(poly_attenuation(np.array([diameter_cm]))[0])

# Flat field is incident intensity I0; raw image is transmitted intensity I.
floor = np.finfo(np.float32).tiny
atten = -np.log(np.maximum(raw, floor) / np.maximum(flat, floor))
row = NY // 2
center_col = NX // 2
measured_center = float(atten[row, center_col])
absolute_error = measured_center - expected_center
relative_error_pct = 100.0 * absolute_error / expected_center

# Map each detector column to the sphere chord length at the object plane.
detector_x_mm = (np.arange(NX) - center_col) * PIXEL_MM
object_x_mm = detector_x_mm / MAGNIFICATION
chord_mm = np.where(np.abs(object_x_mm) <= RADIUS_MM,
                    2.0 * np.sqrt(np.maximum(RADIUS_MM**2 - object_x_mm**2, 0.0)), 0.0)
poly_profile = poly_attenuation(chord_mm / 10.0)

print(f'Material: aluminium, density = {DENSITY_G_CM3:.3f} g/cm^3')
print('Spectrum bins (keV):', energies_kev.tolist())
print(f'Central path length: {DIAMETER_MM:.3f} mm = {diameter_cm:.4f} cm')
print(f'Spectrum-integrated expected −ln(I/I0): {expected_center:.6f}')
print(f'Measured −ln(I/I0), central pixel: {measured_center:.6f}')
print(f'Central-pixel relative error: {relative_error_pct:+.3f}%')

# Plot effective attenuation against material thickness.
lengths_cm = np.linspace(0.01, diameter_cm, 200)
attenuation_vs_length = poly_attenuation(lengths_cm)
mu_effective = attenuation_vs_length / lengths_cm

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(object_x_mm, atten[row], 'o', ms=3, label='gVXR measured −ln(I/I₀)')
axes[0].plot(object_x_mm, poly_profile, '-', lw=2, label='Spectrum-integrated reference')
axes[0].set_xlabel('Object-space offset from sphere center (mm)')
axes[0].set_ylabel('Attenuation, −ln(I/I₀)')
axes[0].set_title('Projection line through the sphere')
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].plot(lengths_cm * 10.0, mu_effective, color='darkorange')
axes[1].set_xlabel('Aluminium path length (mm)')
axes[1].set_ylabel('Effective attenuation, A/L (cm⁻¹)')
axes[1].set_title('Effective μ changes with thickness (beam hardening)')
axes[1].grid(alpha=0.25)
fig.suptitle('Polychromatic aluminium sphere validation')
fig.tight_layout()
plt.show()


## 5. Save the validation result
The JSON records the supplied spectrum, independently calculated xraylib attenuation at each energy, spectrum-integrated center reference, measured center value, renderer, and relative error. Deviations can arise from mesh tessellation, pixel sampling, or differences between the simulator and reference material models.

In [ ]:
result = {
    'validation': 'polychromatic Beer-Lambert central ray through aluminium sphere',
    'material': 'Al', 'density_g_cm3': DENSITY_G_CM3,
    'spectrum_relative_photon_weights': SPECTRUM_WEIGHTS,
    'spectrum_bins_keV': energies_kev.tolist(),
    'sphere_diameter_mm': DIAMETER_MM, 'central_path_length_cm': diameter_cm,
    'mass_attenuation_cm2_g_xraylib': mass_mu_cm2_g.tolist(),
    'linear_attenuation_cm_inv': mu_bins_cm_inv.tolist(),
    'expected_spectrum_integrated_minus_ln_I_over_I0': expected_center,
    'measured_minus_ln_I_over_I0': measured_center,
    'relative_error_percent': float(relative_error_pct),
    'geometry': {'beam': cfg.beam, 'SOD_cm': SOD_CM, 'SDD_cm': SOD_CM + ODD_CM,
                 'magnification': MAGNIFICATION, 'detector_pixel_mm': PIXEL_MM,
                 'projection_shape_y_x': list(raw.shape)},
    'renderer': core.renderer_info(),
    'renderer_selection': core.renderer_selection_info(),
    'runtime': {'python': sys.version.split()[0], 'platform': platform.platform()}
}
out = Path('gvxr_aluminium_polychromatic_validation.json')
out.write_text(json.dumps(result, indent=2))
print(out.read_text())
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('Saved locally:', out.resolve())
